# Chapter 09: Full FSD Capstone & Real-Time System Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/09_full_fsd_system_architecture.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *How do production systems integrate 8 cameras, 5 neural networks, and 2 actuators into an 80 Hz fail-safe pipeline?*

---

## 1. 🚨 The Real-World Dilemma
At 65 mph, a 200ms latency equals 5.8 meters of blind motion before brakes bite. Production systems use 3 multi-rate tiers (Control at 100-200 Hz, Perception at 30-50 Hz, Route Planning at 10 Hz) with zero-copy ring buffers and watchdog fail-safes.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

latencies = []
cross_track_errors = []

for step in range(100):
    t0 = time.perf_counter()
    time.sleep(0.005) # Perception
    time.sleep(0.003) # Planning
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    latencies.append(elapsed_ms)
    cte = 0.4 * np.exp(-step / 20.0) + np.random.normal(0, 0.01)
    cross_track_errors.append(cte)

print(f"Mean Pipeline Latency: {np.mean(latencies):.2f} ms (Target: <30ms)")
print(f"99th Percentile Latency: {np.percentile(latencies, 99):.2f} ms")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(latencies, color="#58a6ff", lw=1.5)
plt.axhline(30.0, color="r", linestyle="--", label="30ms Deadline")
plt.title("Sensor-to-Actuation Latency Profile")
plt.xlabel("Step")
plt.ylabel("Latency (ms)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(cross_track_errors, color="#3fb950", lw=2)
plt.title("Closed-Loop Cross-Track Error (m)")
plt.xlabel("Step")
plt.ylabel("CTE (m)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **Random steering jolts every few minutes** | Python Garbage Collection (GC) pauses threads for 40ms. | Profile GC pause durations. | Disable automatic GC in 100 Hz control loop; pre-allocate tensors. |
| **Car drifts toward lane edges during heavy loads** | Pipeline latency exceeds 30ms time slice, delivering stale commands. | Measure 99th percentile latency. | Enforce hard 20ms timeout anytime algorithm. |